# 이메일 초안 작성 에이전트

수신자, 목적, 톤을 입력하면 자동으로 이메일 초안을 작성해주는 단일 목적 에이전트입니다.

## 주요 기능
- 수신자 및 목적 기반 이메일 초안 생성
- 톤(친한 교수님/웃어른 / 안친한 교수님 / 친구) 조정
- 초안 수정 요청 처리

In [1]:
%pip install -Uq langchain langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()


@tool
def generate_email(recipient: str, purpose: str, tone: str, additional: str = "") -> str:
    """수신자, 목적, 톤을 입력받아 이메일 초안을 생성합니다.

    Args:
        recipient: 이메일 수신자 (예: 홍길동 교수님, 친구 이름)
        purpose: 이메일 목적 (예: 과제 기한 연장 요청, 점심 약속 제안)
        tone: 이메일 톤 - '친한 교수님/웃어른', '안친한 교수님', '친구' 중 하나
        additional: 추가 요구사항 (선택사항)

    Returns:
        str: 작성된 이메일 초안
    """
    return f"""
다음 조건으로 이메일 초안을 작성해주세요:
- 수신자: {recipient}
- 목적: {purpose}
- 톤: {tone}
{f'- 추가 요구사항: {additional}' if additional else ''}

톤 가이드:
- 친한 교수님/웃어른: 존댓말을 사용하되 너무 딱딱하지 않은 적당한 격식체
- 안친한 교수님: 격식체, 비즈니스 메일 작성법을 철저히 따름
- 친구: 반말, 편하고 자연스러운 표현

이메일에 반드시 포함할 요소: 제목(Subject), 인사말, 본문, 마무리 인사, 서명
"""


@tool
def revise_email(original_email: str, revision_request: str) -> str:
    """기존 이메일 초안과 수정 요청을 입력받아 수정된 이메일을 생성합니다.

    Args:
        original_email: 수정할 기존 이메일 초안
        revision_request: 수정 요청사항 (예: 더 간결하게, 사과 표현 강하게)

    Returns:
        str: 수정된 이메일 초안
    """
    return f"""
아래 이메일 초안을 수정 요청에 맞게 수정해주세요.

[기존 이메일]
{original_email}

[수정 요청]
{revision_request}
"""


model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7
)

agent = create_agent(
    model=model,
    tools=[generate_email, revise_email]
)

### 예시: 초안 수정 요청

In [3]:
# 1단계: 이메일 초안 생성
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "홍길동 교수님께 과제 제출 기한 3일 연장 요청 이메일 써줘. 안친한 교수님 톤으로, 갑작스러운 건강 악화 이유야."
        }
    ]
})

draft = response['messages'][-1].content  # ← 이게 빠진 거예요!
print(draft)

# 2단계: 초안 수정 요청
response2 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"이 이메일을 더 간결하게 줄이고 사과 표현을 더 강하게 수정해줘:\n\n{draft}"
        }
    ]
})

print(response2['messages'][-1].content)

**Subject:** 과제 제출 기한 연장 요청드립니다  

홍길동 교수님께  

존경하는 교수님,  

저는 귀 과목 수강생 **[이름]**(학번 **[학번]**)입니다.  
갑작스러운 건강 악화로 인해 현재 복지센터에서 치료를 받고 있는 상황이며, 의료진의 권고에 따라 최소 3일간 휴식이 필요합니다.  

이에 따라, **[과제명]** 과제의 제출 기한을 현재 예정일(YYYY-MM-DD)에서 3일 연장해 주실 수 있는지 정중히 요청드립니다.  
연장 기간 동안 최선을 다해 과제를 완성하도록 하겠습니다.  

교수님께 불편을 드려 대단히 죄송합니다.  
연장 허락해 주신다면 진심으로 감사드리겠습니다.  

감사합니다.  

[이름]  
[학과]  
[학번]  
[연락처]
**Subject:** 과제 제출 기한 연장 요청

홍길동 교수님께  

존경하는 교수님,  

저는 **[이름]**(학번 **[학번]**)입니다.  
갑작스러운 건강 악화로 복지센터에서 치료 중이며, 의료진의 권고에 따라 최소 3일간 휴식이 필요합니다.  

이에 **[과제명]** 과제 제출 기한을 현재 예정일(YYYY-MM-DD)에서 3일 연장해 주실 수 있는지 정중히 요청드립니다.  
연장 기간 동안 최선을 다해 과제를 완성하겠습니다.  

교수님께 큰 불편을 끼쳐 드려 진심으로 사과드립니다.  
연장 허락해 주신다면 깊이 감사드리겠습니다.  

감사합니다.  

[이름]  
[학과]  
[학번]  
[연락처]
